# JD.com Search and Shopping Behavior Analysis

## Project Objective

This project analyzes customer search and shopping interaction data from the JDsearch dataset to identify patterns in product discovery, engagement, and purchasing activity.

The analysis focuses on how recorded positive interactions are distributed across products, categories, brands, shops, and search queries. The goal is to identify high-performing segments as well as areas with substantial engagement but relatively low purchase representation that may warrant further investigation.

Because the dataset does not provide linked customer journeys, timestamps, or non-interaction impressions, the analysis is descriptive rather than causal. Results are intended to support prioritization and follow-up investigation rather than demonstrate that a particular factor caused a purchase.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Project environment is ready!")

## Dataset Scope

The analytical dataset contains recorded positive customer interactions associated with search results and products. The interaction labels represent:

- **1 — Click**
- **2 — Add to Cart**
- **3 — Purchase**

Records representing no interaction are not included in the processed dataset. Therefore, the analysis does not measure traditional conversion from impressions, visitors, or sessions.

The primary metric used throughout the project is **Purchase Share**, defined as:

**Purchases ÷ Recorded Positive Interactions**

Purchase Share should not be interpreted as a conventional e-commerce conversion rate.

The dataset also contains anonymized identifiers for products, brands, shops, categories, queries, and interaction records. Search-query content is tokenized, which limits semantic interpretation of individual searches.

In [ ]:
from pathlib import Path
input_path = Path("processed_with_meta.parquet")
if not input_path.exists():
    raise FileNotFoundError("Place processed_with_meta.parquet in the notebook working directory before running.")
df = pd.read_parquet(input_path)
print("Rows and columns:", df.shape)
df.head()


## Data Validation

Before transforming the dataset, I reviewed its structure, data types, summary statistics, and missing-value patterns. These checks were used to verify that the imported data was suitable for analysis and to identify fields requiring renaming, type conversion, or additional validation.

The validation process includes:

- reviewing dataset dimensions and sample records;
- inspecting column data types;
- checking descriptive statistics;
- identifying missing values; and
- verifying the structure of identifier and interaction fields.

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.describe(include='all')

In [ ]:
df.isnull().sum()

## Data Cleaning and Preparation

A separate working DataFrame was created so that the imported source data remained unchanged during transformation.

The cleaning process standardizes column names, converts identifier fields to consistent data types, validates relationships between source identifiers, and creates analysis-friendly interaction fields.

Several decisions are especially important to the analysis:

- The source candidate-product identifier was compared with `product_id` before the redundant field was removed.
- Interaction labels were mapped to descriptive categories: Click, Add to Cart, and Purchase.
- Binary fields were created to support aggregation of purchases and cart-or-purchase interactions.
- Product, brand, category, shop, query-record, and candidate-record identifiers were standardized for consistent grouping and comparison.

In [ ]:
clean_df = df.copy()

clean_df =clean_df.rename(columns={
    "candidates": "candidate_product_id",
    "labels": "interaction_label",
    "c_id": "candidate_record_id",
    "q_id": "query_record_id",
    "wid": "product_id",
    "name": "product_name_tokens",
    "cate_id_4": "category_level_4_id"
})

clean_df.head()


In [ ]:
product_ids_match = (
    clean_df["candidate_product_id"] == clean_df["product_id"]
).all()

print("Do candidate_product_id and product_id always match?", product_ids_match)

In [ ]:
clean_df = clean_df.drop(columns=["candidate_product_id"])

In [ ]:
interaction_mapping = {
    1: "Click",
    2: "Add to Cart",
    3: "Purchase"
}

clean_df["interaction_label"] = clean_df["interaction_label"].astype("int8")

clean_df["interaction_type"] = (
    clean_df["interaction_label"]
    .map(interaction_mapping)
)

clean_df["added_to_cart_or_purchased"] = (
    clean_df["interaction_label"] >= 2
).astype("int8")

clean_df["purchased"] = (
    clean_df["interaction_label"] == 3
).astype("int8")

clean_df[
    [
        "interaction_label",
        "interaction_type",
        "added_to_cart_or_purchased",
        "purchased"
    ]
].head(10)

In [ ]:
id_columns = [
    "product_id",
    "brand_id",
    "category_level_4_id",
    "shop_id",
    "candidate_record_id",
    "query_record_id"
]

for column in id_columns:
    clean_df[column] = clean_df[column].astype("string")

clean_df.dtypes

In [ ]:
print(clean_df.columns.tolist())

In [ ]:
interaction_summary = (
    clean_df["interaction_type"]
    .value_counts()
    .reindex(["Click", "Add to Cart", "Purchase"], fill_value=0)
    .rename_axis("interaction_type")
    .reset_index(name="interaction_count")
)

interaction_summary["percentage"] = (
    interaction_summary["interaction_count"]
    / interaction_summary["interaction_count"].sum()
    * 100
).round(2)

interaction_summary

In [ ]:
print("Total interactions:", len(clean_df))
print("Unique query records:", clean_df["query_record_id"].nunique())
print("Unique products:", clean_df["product_id"].nunique())
print("Unique brands:", clean_df["brand_id"].nunique())
print("Unique categories:", clean_df["category_level_4_id"].nunique())
print("Unique shops:", clean_df["shop_id"].nunique())
print("Duplicate rows:", clean_df.duplicated().sum())

## Duplicate-Row Sensitivity Analysis

The cleaned dataset contains 16,575 exact duplicate rows, representing approximately 5.18% of the 320,132 recorded interactions. These records were not automatically removed because the source data does not include timestamps or a unique event-level identifier. As a result, identical rows cannot be conclusively classified as accidental duplicates rather than legitimate repeated customer interactions.

To evaluate whether retaining these records materially affects the analysis, a sensitivity test was performed by creating a deduplicated version of the dataset and recalculating the headline Purchase Share metric. The deduplicated dataset is used only for comparison; the original cleaned dataset remains the basis for the primary analysis.

In [ ]:
# Duplicate-row sensitivity analysis

total_rows = len(clean_df)
duplicate_rows = clean_df.duplicated().sum()
duplicate_pct = duplicate_rows / total_rows * 100

# Create a comparison dataset with exact duplicate rows removed
deduped_df = clean_df.drop_duplicates()

rows_after_deduplication = len(deduped_df)

# Calculate Purchase Share before and after deduplication
purchase_share_original = (
    clean_df["purchased"].sum() / len(clean_df) * 100
)

purchase_share_deduped = (
    deduped_df["purchased"].sum() / len(deduped_df) * 100
)

purchase_share_difference = (
    purchase_share_deduped - purchase_share_original
)

print(f"Original rows: {total_rows:,}")
print(f"Exact duplicate rows: {duplicate_rows:,}")
print(f"Duplicate percentage: {duplicate_pct:.2f}%")
print(f"Rows after removing exact duplicates: {rows_after_deduplication:,}")
print(f"Original Purchase Share: {purchase_share_original:.2f}%")
print(f"Purchase Share after deduplication: {purchase_share_deduped:.2f}%")
print(
    f"Absolute change in Purchase Share: "
    f"{purchase_share_difference:.2f} percentage points"
)

In [ ]:
clean_df.to_parquet(
    "jdsearch_cleaned_interactions.parquet",
    index=False
)

print("Cleaned dataset saved successfully.")

### Interpretation

Removing all exact duplicate rows reduces the dataset from 320,132 to 303,557 records, a decrease of approximately 5.18%. However, overall Purchase Share changes only from approximately 6.50% to 6.63%, an absolute difference of about 0.13 percentage points.

This sensitivity test indicates that the presence of exact duplicate rows does not materially change the project's headline Purchase Share result. Because the source data does not provide enough information to distinguish accidental duplicate records from legitimate repeated interactions, the duplicates were retained in the primary dataset rather than removed without sufficient evidence.

This approach preserves the original source observations while demonstrating that the main analytical interpretation remains stable under an alternative deduplication assumption.

### Metric Definitions

The following measures are used throughout the analysis:

**Total Interactions**  
The number of recorded positive interaction rows associated with a segment.

**Clicks**  
Records with an interaction label of 1.

**Cart Additions**  
Records with an interaction label of 2.

**Purchases**  
Records with an interaction label of 3.

**Purchase Share**  
Purchases divided by total recorded positive interactions for the relevant segment.

Purchase Share is used as a descriptive composition metric. Because the dataset excludes non-interactions and does not contain linked customer sessions, it should not be interpreted as a traditional conversion rate or funnel-completion rate.

### Analytical Screening Rules

These are exploratory prioritization rules, not statistical significance tests.

- **20+ interactions:** broad ranking screen for sparse product/query analyses and SQL exploratory rankings.
- **100+ interactions:** higher-volume notebook/dashboard opportunity screens.
- **Above-average volume + below overall Purchase Share:** SQL diagnostic screen for active segments with below-benchmark purchase representation.

None of these screens establishes causation, abandonment, or lost revenue.


## Interaction Analysis

The first analytical step examines the overall composition of recorded interactions.

This provides a baseline for understanding how frequently clicks, cart additions, and purchases appear in the processed dataset. Because these records are not linked sequential customer events, the differences between the three interaction types should not be interpreted as funnel abandonment.

Instead, the distribution describes the relative composition of positive interactions available for analysis.

In [ ]:
interaction_summary = clean_df["interaction_type"].value_counts().rename_axis("interaction_type").reset_index(name="interaction_count")

plt.figure(figsize=(8, 5))
chart = sns.barplot(data=interaction_summary, x="interaction_type", y="interaction_count", color="#4C78A8")
chart.bar_label(chart.containers[0], fmt="%.0f", padding=3)
chart.margins(y=0.1)

plt.title("JD.com Recorded Customer Interactions")
plt.xlabel("Interaction Type")
plt.ylabel("Number of Interactions")
plt.ticklabel_format(style="plain", axis="y")
plt.tight_layout()
plt.show()

In [ ]:
interaction_summary.plot.bar(x="interaction_type", y="interaction_count", legend=False)

### Interaction Analysis — Key Finding

Clicks account for the majority of recorded positive interactions, followed by cart additions and purchases. Purchases represent approximately 6.50% of the analyzed interaction records.

This establishes the project's overall Purchase Share baseline and provides a reference point for comparing categories, brands, shops, products, and queries.

## Category Analysis

Category-level analysis evaluates differences in customer activity and purchase representation across product categories.

For each category, the analysis calculates total interactions, clicks, cart additions, purchases, unique products, and Purchase Share. Categories are then examined from two perspectives:

1. categories generating the highest number of purchases; and
2. categories with meaningful interaction volume but relatively low Purchase Share.

A minimum interaction threshold is used when identifying potential opportunities so that very small groups do not receive disproportionate attention.

In [ ]:
category_performance = (
    clean_df.groupby("category_level_4_id")
    .agg(
        total_interactions=("interaction_label", "size"),
        clicks=("interaction_label", lambda x: (x == 1).sum()),
        cart_additions=("interaction_label", lambda x: (x == 2).sum()),
        purchases=("interaction_label", lambda x: (x == 3).sum()),
        unique_products=("product_id", "nunique")
    )
    .reset_index()
)

category_performance["purchase_share_percent"] = (
    category_performance["purchases"]
    / category_performance["total_interactions"]
    * 100
).round(2)

category_performance.head()

In [ ]:
top_categories_by_purchases = (
    category_performance
    .sort_values("purchases", ascending=False)
    .head(15)
)

top_categories_by_purchases

In [ ]:
chart_data = top_categories_by_purchases.sort_values(
    "purchases",
    ascending=True
)

plt.figure(figsize=(10, 7))

chart = sns.barplot(
    data=chart_data,
    x="purchases",
    y="category_level_4_id",
    color="#59A14F"
)

chart.bar_label(chart.containers[0], fmt="%.0f", padding=3)

plt.title("Top 15 Product Categories by Recorded Purchases")
plt.xlabel("Number of Purchases")
plt.ylabel("Anonymized Category ID")
plt.tight_layout()
plt.show()

In [ ]:
high_opportunity_categories = (
    category_performance[
        category_performance["total_interactions"] >= 100
    ]
    .sort_values(
        ["purchase_share_percent", "purchases"],
        ascending=[False, False]
    )
    .head(15)
)

high_opportunity_categories

### Category Analysis — Interpretation

Category performance is distributed across a large number of anonymized categories rather than being concentrated in only a few segments. High interaction volume does not always correspond with high Purchase Share.

Categories with substantial engagement but comparatively low Purchase Share should therefore be treated as investigation priorities rather than assumed performance failures. Additional information on pricing, availability, search ranking, product content, and linked customer journeys would be needed to determine the underlying cause.

## Brand and Shop Analysis

The next stage compares interaction and purchase activity across anonymized brands and shops.

Brand-level metrics are calculated using the same framework as category analysis. Shop-level analysis excludes the `-1` shop identifier, which represents an invalid or unavailable shop value rather than a meaningful seller.

The analysis identifies both strong performers and segments with sufficient activity but relatively low Purchase Share. These results can help prioritize brands or sellers for deeper investigation.

In [ ]:
brand_performance = (
    clean_df.groupby("brand_id")
    .agg(
        total_interactions=("interaction_label", "size"),
        clicks=("interaction_label", lambda x: (x == 1).sum()),
        cart_additions=("interaction_label", lambda x: (x == 2).sum()),
        purchases=("interaction_label", lambda x: (x == 3).sum()),
        unique_products=("product_id", "nunique")
    )
    .reset_index()
)

brand_performance["purchase_share_percent"] = (
    brand_performance["purchases"]
    / brand_performance["total_interactions"]
    * 100
).round(2)

brand_performance.head()

In [ ]:
top_brands_by_purchases = (
    brand_performance
    .sort_values("purchases", ascending=False)
    .head(15)
)

top_brands_by_purchases

In [ ]:
chart_data = top_brands_by_purchases.sort_values(
    "purchases",
    ascending=True
)

plt.figure(figsize=(10, 7))

chart = sns.barplot(
    data=chart_data,
    x="purchases",
    y="brand_id",
    color="#E15759"
)

chart.bar_label(chart.containers[0], fmt="%.0f", padding=3)

plt.title("Top 15 Brands by Recorded Purchases")
plt.xlabel("Number of Purchases")
plt.ylabel("Anonymized Brand ID")
plt.tight_layout()
plt.show()

In [ ]:
high_opportunity_brands = (
    brand_performance[
        brand_performance["total_interactions"] >= 100
    ]
    .sort_values(
        ["purchase_share_percent", "purchases"],
        ascending=[False, False]
    )
    .head(15)
)

high_opportunity_brands

In [ ]:
shop_performance = (
    clean_df[clean_df["shop_id"] != "-1"]
    .groupby("shop_id")
    .agg(
        total_interactions=("interaction_label", "size"),
        clicks=("interaction_label", lambda x: (x == 1).sum()),
        cart_additions=("interaction_label", lambda x: (x == 2).sum()),
        purchases=("interaction_label", lambda x: (x == 3).sum()),
        unique_products=("product_id", "nunique"),
        unique_brands=("brand_id", "nunique")
    )
    .reset_index()
)

shop_performance["purchase_share_percent"] = (
    shop_performance["purchases"]
    / shop_performance["total_interactions"]
    * 100
).round(2)

shop_performance.head()

In [ ]:
top_shops_by_purchases = (
    shop_performance
    .sort_values("purchases", ascending=False)
    .head(15)
)

top_shops_by_purchases

In [ ]:
chart_data = top_shops_by_purchases.sort_values(
    "purchases",
    ascending=True
)

plt.figure(figsize=(10, 7))

chart = sns.barplot(
    data=chart_data,
    x="purchases",
    y="shop_id",
    color="#B279A2"
)

chart.bar_label(chart.containers[0], fmt="%.0f", padding=3)

plt.title("Top 15 Shops by Recorded Purchases")
plt.xlabel("Number of Purchases")
plt.ylabel("Anonymized Shop ID")
plt.tight_layout()
plt.show()


In [ ]:
shops_needing_attention = (
    shop_performance[
        shop_performance["total_interactions"] >= 100
    ]
    .sort_values(
        ["purchase_share_percent", "total_interactions"],
        ascending=[True, False]
    )
    .head(15)
)

shops_needing_attention

### Brand and Shop Analysis — Interpretation

The results show that engagement and purchase representation vary considerably across brands and sellers. High activity alone does not guarantee a high proportion of purchases.

These differences are descriptive and should not be interpreted as evidence that a brand or seller caused stronger or weaker purchasing behavior. Factors such as product mix, pricing, availability, search visibility, and customer intent may contribute to the observed differences.

## Product Analysis

Product-level analysis examines which individual products generate the most purchase activity and which products may warrant additional investigation.

Metrics include total interactions, clicks, cart additions, purchases, Purchase Share, associated brand, category, shop, and query activity.

Product-level results require additional caution because many products appear only a small number of times in the dataset. A minimum interaction threshold is therefore used when identifying potential underperforming products to reduce the influence of extremely sparse records.

In [ ]:
product_performance = (
    clean_df.groupby(
        [
            "product_id",
            "brand_id",
            "category_level_4_id",
            "shop_id"
        ],
        dropna=False
    )
    .agg(
        total_interactions=("interaction_label", "size"),
        clicks=("interaction_label", lambda x: (x == 1).sum()),
        cart_additions=("interaction_label", lambda x: (x == 2).sum()),
        purchases=("interaction_label", lambda x: (x == 3).sum()),
        unique_query_records=("query_record_id", "nunique")
    )
    .reset_index()
)

product_performance["purchase_share_percent"] = (
    product_performance["purchases"]
    / product_performance["total_interactions"]
    * 100
).round(2)

product_performance.head()

product_performance["purchase_share_percent"] = (
    product_performance["purchases"]
    / product_performance["total_interactions"]
    * 100
).round(2)

product_performance.head()

In [ ]:
top_products_by_purchases = (
    product_performance
    .sort_values(
        ["purchases", "total_interactions"],
        ascending=[False, False]
    )
    .head(15)
)

top_products_by_purchases


In [ ]:
chart_data = top_products_by_purchases.sort_values(
    "purchases",
    ascending=True
)

plt.figure(figsize=(10, 7))

chart = sns.barplot(
    data=chart_data,
    x="purchases",
    y="product_id",
    color="#76B7B2"
)

chart.bar_label(chart.containers[0], fmt="%.0f", padding=3)

plt.title("Top 15 Products by Recorded Purchases")
plt.xlabel("Number of Purchases")
plt.ylabel("Anonymized Product ID")
plt.tight_layout()
plt.show()

In [ ]:
products_needing_attention = (
    product_performance[
        product_performance["total_interactions"] >= 20
    ]
    .sort_values(
        ["purchase_share_percent", "total_interactions"],
        ascending=[True, False]
    )
    .head(20)
)

products_needing_attention

In [ ]:
cart_opportunity_products = (
    product_performance[
        product_performance["cart_additions"] >= 10
    ]
    .assign(
        cart_purchase_share=lambda x: (
            x["purchases"]
            / (x["cart_additions"] + x["purchases"])
            * 100
        ).round(2)
    )
    .sort_values(
        ["cart_purchase_share", "cart_additions"],
        ascending=[True, False]
    )
    .head(20)
)

cart_opportunity_products

### Product Analysis — Interpretation

Product-level performance is highly sparse, so rankings based on very small numbers of interactions should not be treated as reliable indicators of product quality or customer preference.

Products with meaningful activity and comparatively low Purchase Share provide stronger candidates for investigation. Possible areas for follow-up include product-page information, price competitiveness, inventory availability, shipping conditions, and search-result positioning.

## Query Analysis

Query-level analysis examines the relationship between anonymized search queries and recorded customer interactions.

For each tokenized query, the analysis calculates total interactions, clicks, cart additions, purchases, unique products, and Purchase Share. Queries are reviewed by overall activity, purchase volume, relatively low Purchase Share, and relatively high purchase representation.

Because query text is tokenized, this analysis can identify which query IDs deserve further investigation but cannot directly explain the customer's search intent.

In [ ]:
query_performance = (
    clean_df.groupby(
        "query_record_id",
        dropna=False
    )
    .agg(
        total_interactions=("interaction_label", "size"),
        clicks=("interaction_label", lambda x: (x == 1).sum()),
        cart_additions=("interaction_label", lambda x: (x == 2).sum()),
        purchases=("interaction_label", lambda x: (x == 3).sum()),
        unique_products=("product_id", "nunique")
    )
    .reset_index()
)

query_performance["purchase_share_percent"] = (
    query_performance["purchases"]
    / query_performance["total_interactions"]
    * 100
).round(2)

query_performance.head()

In [ ]:
top_queries_by_activity = (
    query_performance
    .sort_values("total_interactions", ascending=False)
    .head(15)
)

top_queries_by_activity

In [ ]:
top_queries_by_purchases = (
    query_performance
    .sort_values(
        ["purchases", "total_interactions"],
        ascending=[False, False]
    )
    .head(15)
)

top_queries_by_purchases

In [ ]:
underserved_queries = (
    query_performance[
        query_performance["total_interactions"] >= 20
    ]
    .sort_values(
        ["purchase_share_percent", "total_interactions"],
        ascending=[True, False]
    )
    .head(20)
)

underserved_queries

In [ ]:
high_intent_queries = (
    query_performance[
        query_performance["total_interactions"] >= 20
    ]
    .sort_values(
        ["purchase_share_percent", "purchases"],
        ascending=[False, False]
    )
    .head(20)
)

high_intent_queries

### Query Analysis — Interpretation

Search-query performance varies substantially across the dataset. Some queries generate considerable activity but relatively little purchase representation, while others show stronger purchase composition.

These results may indicate opportunities to investigate search relevance, product-result quality, ranking, availability, or customer intent. However, the anonymized query content prevents direct interpretation of what customers were searching for.

## Limitations

Several limitations affect how these findings should be interpreted.

The dataset includes only positive interaction labels and therefore does not provide impression-level or visitor-level conversion measurement. Individual records are not linked into customer journeys, so clicks, cart additions, and purchases cannot be interpreted as sequential funnel stages.

The data also lacks timestamps, financial measures, pricing, inventory status, shipping information, returns, cancellations, and other contextual variables that could explain differences in purchase behavior.

Product, brand, shop, category, and query identifiers are anonymized, and query content is tokenized. Exact duplicate records are present, but the absence of timestamps or a unique event identifier prevents them from being conclusively classified as erroneous duplicate events. Sensitivity analysis shows that removing them does not materially alter the headline Purchase Share result.

Finally, the analysis is descriptive and exploratory. Observed relationships should be treated as prioritization signals rather than evidence of causation.

## Key Takeaways

The analysis produced several important findings:

- Recorded interactions are dominated by clicks, while purchases account for approximately 6.50% of positive interaction records.
- Purchase activity is distributed across a large number of categories, brands, shops, products, and queries rather than being dominated by only a handful of entities.
- High interaction volume does not consistently correspond with high Purchase Share.
- Segments combining meaningful engagement with relatively low Purchase Share provide useful candidates for deeper investigation.
- Product-level results should be interpreted cautiously because interaction counts are highly sparse for many individual products.
- Duplicate-row sensitivity testing indicates that retaining exact repeated records does not materially alter the headline Purchase Share interpretation.

The findings support prioritizing selected categories, brands, shops, products, and queries for further investigation rather than assuming that observed performance differences have a single cause. Stronger conclusions would require linked customer journeys and additional information such as impressions, timestamps, pricing, inventory, shipping, and financial outcomes.

## Analytical Outputs

The final cleaned interaction dataset and aggregated performance tables are exported for use in PostgreSQL, Tableau, and the project's supporting documentation.

Outputs include category, brand, shop, product, and query performance tables as well as the cleaned interaction-level dataset.

In [ ]:
from pathlib import Path
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
category_performance.to_csv(output_dir / "category_performance.csv", index=False)
brand_performance.to_csv(output_dir / "brand_performance.csv", index=False)
shop_performance.to_csv(output_dir / "shop_performance.csv", index=False)
product_performance.to_csv(output_dir / "product_performance.csv", index=False)
query_performance.to_csv(output_dir / "query_performance.csv", index=False)
print("Canonical analytical tables saved to outputs/.")


In [ ]:
clean_df.to_csv(
    "jdsearch_cleaned_interactions.csv",
    index=False
)

print("CSV created successfully.")


In [ ]:
import os
print(os.path.abspath("jdsearch_cleaned_interactions.csv"))
